# Subsampling & Race Prediction Experiments

### Experiment 1: Subsample White patients to match Black sample size
Retrain all four models (XGBoost, Deep NN, LSTM, Transformer) on a balanced
dataset where White patients are subsampled to ~6,000 to match Black patient count.
Subsampling happens INSIDE each training fold to prevent leakage.
Test sets are never touched. Always evaluated on full test set.

Does the White-Black AUPRC gap persist when sample sizes are equal?
- If gap disappears: sample size was the main driver of disparity
- If gap remains: vital sign patterns and monitoring differences drive the disparity

### Experiment 2: Predict race from vital signs
Train a classifier to predict race group from the same 24x10 time series input.
Run with AND without the missingness mask to isolate whether monitoring
patterns (mask) or vital sign values drive race predictability.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import itertools
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
EXP_DIR  = OUT_DIR / 'subsample_race'
EXP_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000
TARGET_WHITE = 6000   # subsample White to this size per fold training set

# Training hyperparameters identical to original
N_EPOCHS_LSTM   = 50
N_EPOCHS_TRANS  = 60
BATCH_SIZE      = 512
LR_LSTM         = 5e-4
LR_TRANS        = 3e-4

COLORS = {
    'XGBoost':     'steelblue',
    'Deep NN':     'seagreen',
    'LSTM':        'darkorange',
    'Transformer': 'mediumpurple',
}
RACE_COLORS = {
    'White': '#4878CF', 'Black': '#D65F5F',
    'Hispanic': '#6ACC65', 'Asian': '#B47CC7'
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {EXP_DIR}')

---
## 2. Load & align data

In [ ]:
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y        = cohort['hospital_expire_flag'].values.astype(np.int32)
demo     = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values

# Build tabular features (mean/std/min/max x 5 vitals + age)
# Load from events_24h parquet
print('Building tabular features...')
events = pd.read_parquet(OUT_DIR / 'events_24h.parquet')
agg = events.groupby(['stay_id','vital'])['valuenum'].agg(
    ['mean','std','min','max']).unstack('vital')
agg.columns = [f'{vital}_{stat}' for stat, vital in agg.columns]
agg = agg.reset_index()

df_tab = cohort[['stay_id','hospital_expire_flag','anchor_age',
                 'race_group']].merge(agg, on='stay_id', how='inner')

# Align everything to the tabular merge
tab_stay_ids = df_tab['stay_id'].values
tab_mask     = np.isin(
    cohort['stay_id'].values, tab_stay_ids)
cohort_tab   = cohort[tab_mask].reset_index(drop=True)
X_ts_tab     = X_ts[tab_mask]
M_ts_tab     = M_ts[tab_mask]
race_tab     = race_arr[tab_mask]
y_tab        = y[tab_mask]

feature_cols = [c for c in df_tab.columns
                if c not in ['stay_id','hospital_expire_flag','race_group']]
X_tab = df_tab[feature_cols].values.astype(np.float64)

print(f'Cohort: {len(cohort_tab):,}  |  Mortality: {y_tab.mean():.1%}')
print(pd.Series(race_tab).value_counts().to_string())

# Load original baseline OOF probs for comparison
oof_baseline = {
    'XGBoost': np.load(OUT_DIR / 'oof_probs_XGBoost.npy')[kept_idx][tab_mask],
    'Deep NN': np.load(OUT_DIR / 'oof_probs_Deep_NN.npy')[kept_idx][tab_mask],
    'LSTM':    np.load(OUT_DIR / 'oof_probs_LSTM.npy')[kept_idx][tab_mask],
}
# Transformer if available
trans_path = OUT_DIR / 'oof_probs_Transformer.npy'
if trans_path.exists():
    oof_baseline['Transformer'] = np.load(trans_path)[kept_idx][tab_mask]

print('\nBaseline AUPRC (full cohort):')
for m, probs in oof_baseline.items():
    for g in ['Black', 'White']:
        mask = race_tab == g
        ap = average_precision_score(y_tab[mask], probs[mask])
        print(f'  {m:<14} {g:<8} {ap:.4f}')

---
## 3. Fold splits

In [ ]:
strat_label = (pd.Series(race_tab).astype(str) + '_' +
               pd.Series(y_tab).astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(y_tab)), strat_label))
print(f'Fold test sizes: {[len(t) for _,t in fold_splits]}')

---
## 4. Model definitions

In [ ]:
class MortalityLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=128,
                 num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)


class MortalityTransformer(nn.Module):
    def __init__(self, input_size=10, d_model=128, nhead=8,
                 num_layers=3, dropout=0.3, max_len=24):
        super().__init__()
        self.input_proj  = nn.Linear(input_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4, dropout=dropout,
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            enable_nested_tensor=False)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        bs, seq_len, _ = x.shape
        positions = torch.arange(seq_len, device=x.device)
        x = self.input_proj(x) + self.pos_embedding(positions)
        x = self.transformer(self.dropout(x))
        return self.classifier(x.mean(dim=1)).squeeze(1)


class MortalityDNN(nn.Module):
    def __init__(self, input_size=240, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Linear(input_size,512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(dropout))
        self.block2 = nn.Sequential(
            nn.Linear(512,256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(dropout))
        self.block3 = nn.Sequential(
            nn.Linear(256,128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(dropout))
        self.block4 = nn.Sequential(
            nn.Linear(128,64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout))
        self.residual_proj = nn.Linear(256, 64)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x  = x.view(x.size(0), -1)
        x  = self.block1(x)
        x2 = self.block2(x)
        x  = self.block4(self.block3(x2)) + self.residual_proj(x2)
        return self.out(x).squeeze(1)


def train_model(model, train_loader, val_loader, y_te,
                n_epochs, lr, use_cosine=False, label=''):
    scale_pw  = torch.tensor(
        [(y_te==0).sum()/(y_te==1).sum()],
        dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=scale_pw)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-4)
    if use_cosine:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_epochs, eta_min=1e-5)
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', patience=3, factor=0.5)

    best_auprc = 0
    best_probs = None

    for epoch in range(1, n_epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if use_cosine: scheduler.step()

        model.eval()
        probs_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                probs_list.append(
                    torch.sigmoid(model(xb.to(device))).cpu().numpy())
        probs = np.concatenate(probs_list)
        ap    = average_precision_score(y_te, probs)
        if not use_cosine: scheduler.step(ap)
        if ap > best_auprc:
            best_auprc = ap
            best_probs = probs.copy()
        if epoch % 10 == 0 or epoch == 1:
            print(f'    [{label}] ep {epoch:02d}  AUPRC={ap:.4f}'
                  f'{"  *" if ap==best_auprc else ""}')

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_probs, best_auprc


def normalise(X_input, X_train):
    ts_mean = X_train.mean(axis=(0,1), keepdims=True)
    ts_std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std


def build_ts_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)


print('All model classes defined.')

---
## 5. Experiment 1: Subsample White to 6k and retrain all four models

In [ ]:
def subsample_train(train_idx, target_white=TARGET_WHITE, seed=42):
    """
    Within a training fold, subsample White patients to target_white.
    All other race groups kept in full.
    Returns subsampled training indices.
    """
    rng = np.random.default_rng(seed)
    white_idx = train_idx[race_tab[train_idx] == 'White']
    other_idx = train_idx[race_tab[train_idx] != 'White']

    n_white = min(target_white, len(white_idx))
    white_sub = rng.choice(white_idx, size=n_white, replace=False)

    sub_idx = np.concatenate([white_sub, other_idx])
    sub_idx = np.sort(sub_idx)
    return sub_idx


# Sanity check on fold 0
train0, test0 = fold_splits[0]
sub0 = subsample_train(train0)
print('Fold 0 training composition after subsampling:')
for g in KEEP_RACES:
    n_orig = (race_tab[train0] == g).sum()
    n_sub  = (race_tab[sub0]   == g).sum()
    print(f'  {g:<10}  original={n_orig:,}  subsampled={n_sub:,}')

In [ ]:
print('Running Experiment 1 — Subsampled White (~6k) retraining')
print('All four models  |  5-fold CV')
print('=' * 65)

oof_sub = {m: np.zeros(len(y_tab)) for m in ['XGBoost','Deep NN','LSTM','Transformer']}

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────')

    # Subsample training set
    sub_idx  = subsample_train(train_idx, seed=RANDOM_STATE + fold)
    y_tr     = y_tab[sub_idx]
    y_te     = y_tab[test_idx]
    race_tr  = race_tab[sub_idx]

    print(f'  Train size after subsampling: {len(sub_idx):,}')
    for g in KEEP_RACES:
        print(f'    {g}: {(race_tr==g).sum():,}')

    scale_pw = (y_tr==0).sum() / (y_tr==1).sum()

    # XGBoost 
    print('  [XGBoost]')
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    X_tr_tab = scl.fit_transform(imp.fit_transform(X_tab[sub_idx]))
    X_te_tab = scl.transform(imp.transform(X_tab[test_idx]))

    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0, device='cuda')
    xgb.fit(X_tr_tab, y_tr)
    xgb_p = xgb.predict_proba(X_te_tab)[:,1]
    oof_sub['XGBoost'][test_idx] = xgb_p
    print(f'    AUPRC={average_precision_score(y_te, xgb_p):.4f}')

    # Prepare TS inputs 
    X_tr_ts = X_ts_tab[sub_idx]
    M_tr_ts = M_ts_tab[sub_idx]
    X_te_ts = X_ts_tab[test_idx]
    M_te_ts = M_ts_tab[test_idx]

    X_tr_n  = normalise(X_tr_ts, X_tr_ts)
    X_te_n  = normalise(X_te_ts, X_tr_ts)
    X_tr_in = build_ts_input(X_tr_n, M_tr_ts)
    X_te_in = build_ts_input(X_te_n, M_te_ts)

    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_in), y_tr_t),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True)
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_te_in), y_te_t),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True)

    # Deep NN 
    print('  [Deep NN]')
    dnn = MortalityDNN(input_size=240, dropout=0.3).to(device)
    if n_gpus > 1: dnn = nn.DataParallel(dnn)
    dnn_p, _ = train_model(
        dnn, train_loader, val_loader, y_te,
        n_epochs=80, lr=3e-4, use_cosine=True, label='DNN')
    oof_sub['Deep NN'][test_idx] = dnn_p

    # LSTM 
    print('  [LSTM]')
    lstm = MortalityLSTM().to(device)
    if n_gpus > 1: lstm = nn.DataParallel(lstm)
    lstm_p, _ = train_model(
        lstm, train_loader, val_loader, y_te,
        n_epochs=N_EPOCHS_LSTM, lr=LR_LSTM,
        use_cosine=False, label='LSTM')
    oof_sub['LSTM'][test_idx] = lstm_p

    # Transformer
    print('  [Transformer]')
    trans = MortalityTransformer().to(device)
    if n_gpus > 1: trans = nn.DataParallel(trans)
    trans_p, _ = train_model(
        trans, train_loader, val_loader, y_te,
        n_epochs=N_EPOCHS_TRANS, lr=LR_TRANS,
        use_cosine=True, label='Transformer')
    oof_sub['Transformer'][test_idx] = trans_p

    # Fold summary
    print(f'\n  Fold {fold+1} summary (Black vs White AUPRC):')
    for m, probs in oof_sub.items():
        if probs[test_idx].sum() == 0: continue
        for g in ['Black', 'White']:
            mask = race_tab[test_idx] == g
            if mask.sum() > 0 and y_te[mask].sum() > 0:
                ap = average_precision_score(y_te[mask], probs[test_idx][mask])
                print(f'    {m:<14} {g:<8} {ap:.4f}')

# Save
for m, probs in oof_sub.items():
    np.save(EXP_DIR / f'oof_sub_{m.replace(" ","_")}.npy', probs)

print('\nExperiment 1 complete. OOF probs saved.')

In [ ]:
# Results: subsampled vs original 
print('AUPRC comparison — original vs subsampled White (6k)')
print('=' * 70)
print(f'{"Model":<14}  {"Condition":<14}  {"Black":>8}  '
      f'{"White":>8}  {"Gap":>8}')
print('-' * 55)

comparison_records = []
for m in ['XGBoost', 'Deep NN', 'LSTM', 'Transformer']:
    for condition, probs_dict in [
        ('Original', oof_baseline),
        ('Sub White 6k', oof_sub)
    ]:
        if m not in probs_dict: continue
        probs = probs_dict[m]
        bl = average_precision_score(
            y_tab[race_tab=='Black'], probs[race_tab=='Black'])
        wh = average_precision_score(
            y_tab[race_tab=='White'], probs[race_tab=='White'])
        gap = wh - bl
        print(f'{m:<14}  {condition:<14}  {bl:>8.4f}  {wh:>8.4f}  {gap:>+8.4f}')
        comparison_records.append({
            'model': m, 'condition': condition,
            'black': bl, 'white': wh, 'gap': gap
        })
    print()

comp_df = pd.DataFrame(comparison_records)
comp_df.to_csv(EXP_DIR / 'subsample_comparison.csv', index=False)
print('Saved subsample_comparison.csv')

In [ ]:
# Figure: Original vs subsampled gap comparison 
models_plot = [m for m in ['XGBoost','Deep NN','LSTM','Transformer']
               if m in oof_sub]
x     = np.arange(len(models_plot))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, group, color_orig, color_sub in [
    (axes[0], 'Black',  '#D65F5F', '#e8a0a0'),
    (axes[1], 'White',  '#4878CF', '#a0b8e8'),
]:
    orig_vals = []
    sub_vals  = []
    for m in models_plot:
        mask = race_tab == group
        orig_vals.append(average_precision_score(
            y_tab[mask], oof_baseline[m][mask]))
        sub_vals.append(average_precision_score(
            y_tab[mask], oof_sub[m][mask]))
    ax.bar(x - width/2, orig_vals, width, label='Original (full White)',
           color=color_orig, edgecolor='white', alpha=0.9)
    ax.bar(x + width/2, sub_vals,  width, label='White subsampled to 6k',
           color=color_sub, edgecolor='white', alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(models_plot, rotation=15)
    ax.set_ylabel('AUPRC')
    ax.set_title(f'{group} patients AUPRC\nOriginal vs subsampled')
    ax.legend(fontsize=8)
    ax.set_ylim(0.3, 0.5)

# Gap comparison
ax = axes[2]
orig_gaps = []
sub_gaps  = []
for m in models_plot:
    bl_o = average_precision_score(
        y_tab[race_tab=='Black'], oof_baseline[m][race_tab=='Black'])
    wh_o = average_precision_score(
        y_tab[race_tab=='White'], oof_baseline[m][race_tab=='White'])
    bl_s = average_precision_score(
        y_tab[race_tab=='Black'], oof_sub[m][race_tab=='Black'])
    wh_s = average_precision_score(
        y_tab[race_tab=='White'], oof_sub[m][race_tab=='White'])
    orig_gaps.append(wh_o - bl_o)
    sub_gaps.append(wh_s - bl_s)

ax.bar(x - width/2, orig_gaps, width, label='Original',
       color='#7f8c8d', edgecolor='white', alpha=0.9)
ax.bar(x + width/2, sub_gaps,  width, label='White subsampled to 6k',
       color='#2ecc71', edgecolor='white', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(models_plot, rotation=15)
ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Disparity gap\nDoes equalising sample size close the gap?')
ax.legend(fontsize=8)
for i, (og, sg) in enumerate(zip(orig_gaps, sub_gaps)):
    ax.text(i-width/2, og+0.001, f'{og:.3f}', ha='center', fontsize=8)
    ax.text(i+width/2, sg+0.001, f'{sg:.3f}', ha='center', fontsize=8)

plt.suptitle('Effect of equalising White sample size to ~6k\n'
             'Does sample size explain the racial performance gap?', fontsize=12)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_subsample_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_subsample_comparison.png')

---
## 6. Experiment 2: Predict race from vital signs

In [ ]:
# Race labels as integers for multiclass classification
race_to_int = {g: i for i, g in enumerate(KEEP_RACES)}
y_race      = np.array([race_to_int[r] for r in race_tab])
# Binary: Black vs rest (most important for thesis)
y_black_vs_rest = (race_tab == 'Black').astype(int)

print('Race label distribution:')
for g, i in race_to_int.items():
    print(f'  {g}: {(y_race==i).sum():,}')

# Stratify on race for CV
skf_race    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)
race_splits = list(skf_race.split(np.zeros(len(y_race)), y_race))

In [ ]:
# Race prediction LSTM 
# We use binary classification: Black vs rest

def run_race_prediction_lstm(use_mask=True, label='with mask'):
    """
    Train LSTM to predict Black vs rest.
    use_mask=True:  input is (N, 24, 10) — vitals + missingness mask
    use_mask=False: input is (N, 24, 5)  — vitals only, no mask
    """
    input_size = 10 if use_mask else 5
    oof_probs  = np.zeros(len(y_black_vs_rest))

    for fold, (train_idx, test_idx) in enumerate(race_splits):
        y_tr = y_black_vs_rest[train_idx]
        y_te = y_black_vs_rest[test_idx]

        X_tr_ts = X_ts_tab[train_idx]
        M_tr_ts = M_ts_tab[train_idx]
        X_te_ts = X_ts_tab[test_idx]
        M_te_ts = M_ts_tab[test_idx]

        X_tr_n = normalise(X_tr_ts, X_tr_ts)
        X_te_n = normalise(X_te_ts, X_tr_ts)

        if use_mask:
            X_tr_in = build_ts_input(X_tr_n, M_tr_ts)
            X_te_in = build_ts_input(X_te_n, M_te_ts)
        else:
            X_tr_in = X_tr_n.astype(np.float32)
            X_te_in = X_te_n.astype(np.float32)

        y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
        y_te_t = torch.tensor(y_te, dtype=torch.float32)

        train_loader = DataLoader(
            TensorDataset(torch.tensor(X_tr_in), y_tr_t),
            batch_size=BATCH_SIZE, shuffle=True,
            num_workers=4, pin_memory=True)
        val_loader = DataLoader(
            TensorDataset(torch.tensor(X_te_in), y_te_t),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=4, pin_memory=True)

        model = MortalityLSTM(input_size=input_size).to(device)
        if n_gpus > 1: model = nn.DataParallel(model)

        # Use AUROC as monitoring metric for race prediction
        scale_pw  = torch.tensor(
            [(y_tr==0).sum()/(y_tr==1).sum()],
            dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=scale_pw)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR_LSTM, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', patience=3, factor=0.5)

        best_auroc = 0
        best_probs_fold = None

        for epoch in range(1, N_EPOCHS_LSTM+1):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            pl = []
            with torch.no_grad():
                for xb, _ in val_loader:
                    pl.append(
                        torch.sigmoid(model(xb.to(device))).cpu().numpy())
            probs = np.concatenate(pl)
            ar    = roc_auc_score(y_te, probs)
            scheduler.step(ar)
            if ar > best_auroc:
                best_auroc = ar
                best_probs_fold = probs.copy()
            if epoch % 10 == 0 or epoch == 1:
                print(f'    [Race LSTM {label}] fold {fold+1} '
                      f'ep {epoch:02d}  AUROC={ar:.4f}'
                      f'{"  *" if ar==best_auroc else ""}')

        oof_probs[test_idx] = best_probs_fold
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return oof_probs


print('Running race prediction — LSTM with missingness mask...')
race_probs_with_mask = run_race_prediction_lstm(
    use_mask=True, label='with mask')
np.save(EXP_DIR / 'race_pred_lstm_with_mask.npy', race_probs_with_mask)

print('\nRunning race prediction — LSTM WITHOUT missingness mask...')
race_probs_no_mask = run_race_prediction_lstm(
    use_mask=False, label='no mask')
np.save(EXP_DIR / 'race_pred_lstm_no_mask.npy', race_probs_no_mask)

print('\nRace prediction AUROC (Black vs rest):')
auroc_with = roc_auc_score(y_black_vs_rest, race_probs_with_mask)
auroc_no   = roc_auc_score(y_black_vs_rest, race_probs_no_mask)
print(f'  With mask:     {auroc_with:.4f}')
print(f'  Without mask:  {auroc_no:.4f}')
print(f'  Drop from mask removal: {auroc_with-auroc_no:+.4f}')

In [ ]:
# Race prediction XGBoost (tabular features) 
print('Running race prediction — XGBoost (tabular features)...')

oof_race_xgb_with = np.zeros(len(y_black_vs_rest))
oof_race_xgb_no   = np.zeros(len(y_black_vs_rest))

for fold, (train_idx, test_idx) in enumerate(race_splits):
    y_tr = y_black_vs_rest[train_idx]
    y_te = y_black_vs_rest[test_idx]
    scale_pw = (y_tr==0).sum() / (y_tr==1).sum()

    # With all features (includes aggregated missingness info indirectly)
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    X_tr_s = scl.fit_transform(imp.fit_transform(X_tab[train_idx]))
    X_te_s = scl.transform(imp.transform(X_tab[test_idx]))

    xgb_with = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0, device='cuda')
    xgb_with.fit(X_tr_s, y_tr)
    oof_race_xgb_with[test_idx] = xgb_with.predict_proba(X_te_s)[:,1]

    # Without std features (std captures missingness indirectly)
    # Use only mean features
    mean_cols = [i for i, c in enumerate(feature_cols) if '_mean' in c or c == 'anchor_age']
    X_tr_mean = scl.fit_transform(imp.fit_transform(X_tab[train_idx][:,mean_cols]))
    X_te_mean = scl.transform(imp.transform(X_tab[test_idx][:,mean_cols]))

    xgb_no = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0, device='cuda')
    xgb_no.fit(X_tr_mean, y_tr)
    oof_race_xgb_no[test_idx] = xgb_no.predict_proba(X_te_mean)[:,1]

    print(f'  Fold {fold+1}  with_all={roc_auc_score(y_te, oof_race_xgb_with[test_idx]):.4f}  '
          f'mean_only={roc_auc_score(y_te, oof_race_xgb_no[test_idx]):.4f}')

print('\nXGBoost race prediction AUROC (Black vs rest):')
print(f'  All features:  {roc_auc_score(y_black_vs_rest, oof_race_xgb_with):.4f}')
print(f'  Mean only:     {roc_auc_score(y_black_vs_rest, oof_race_xgb_no):.4f}')

np.save(EXP_DIR / 'race_pred_xgb_all.npy',  oof_race_xgb_with)
np.save(EXP_DIR / 'race_pred_xgb_mean.npy', oof_race_xgb_no)

In [ ]:
# Figure: Race prediction ROC curves 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

random_baseline = y_black_vs_rest.mean()

# ROC curves
ax = axes[0]
for label, probs, color, ls in [
    ('LSTM + mask', race_probs_with_mask, 'darkorange', '-'),
    ('LSTM no mask', race_probs_no_mask,  'darkorange', '--'),
    ('XGBoost all features', oof_race_xgb_with, 'steelblue', '-'),
    ('XGBoost mean only',    oof_race_xgb_no,   'steelblue', '--'),
]:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_black_vs_rest, probs)
    ar = roc_auc_score(y_black_vs_rest, probs)
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2,
            label=f'{label} (AUROC={ar:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random (0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curve — predicting Black vs rest\nfrom vital signs')
ax.legend(fontsize=8)

# PR curves
ax = axes[1]
for label, probs, color, ls in [
    ('LSTM + mask', race_probs_with_mask, 'darkorange', '-'),
    ('LSTM no mask', race_probs_no_mask,  'darkorange', '--'),
    ('XGBoost all features', oof_race_xgb_with, 'steelblue', '-'),
    ('XGBoost mean only',    oof_race_xgb_no,   'steelblue', '--'),
]:
    prec, rec, _ = precision_recall_curve(y_black_vs_rest, probs)
    ap = average_precision_score(y_black_vs_rest, probs)
    ax.plot(rec, prec, color=color, linestyle=ls, linewidth=2,
            label=f'{label} (AUPRC={ap:.3f})')
ax.axhline(random_baseline, color='gray', linestyle=':',
           label=f'Prevalence ({random_baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR curve — predicting Black vs rest\nfrom vital signs')
ax.legend(fontsize=8)
ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle('Can we predict race from vital signs?\n'
             'Solid = with missingness mask  |  Dashed = without mask',
             fontsize=12)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_race_prediction.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_race_prediction.png')

In [ ]:
# Figure: AUROC bar chart with and without mask 
fig, ax = plt.subplots(figsize=(9, 5))

models_race = [
    'LSTM\n+ mask', 'LSTM\nno mask',
    'XGBoost\nall features', 'XGBoost\nmean only'
]
aurocs = [
    roc_auc_score(y_black_vs_rest, race_probs_with_mask),
    roc_auc_score(y_black_vs_rest, race_probs_no_mask),
    roc_auc_score(y_black_vs_rest, oof_race_xgb_with),
    roc_auc_score(y_black_vs_rest, oof_race_xgb_no),
]
bar_colors = ['darkorange', '#f0c080', 'steelblue', '#a0c0e0']

bars = ax.bar(models_race, aurocs, color=bar_colors,
              edgecolor='white', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.5,
           label='Random baseline (0.500)')
for bar, val in zip(bars, aurocs):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('AUROC (Black vs rest)')
ax.set_title('Race predictability from vital signs\n'
             'How much does the missingness mask contribute?')
ax.legend(fontsize=9)
ax.set_ylim(0.4, max(aurocs)*1.12)

# Annotate the drop from removing mask
drop_lstm = aurocs[0] - aurocs[1]
drop_xgb  = aurocs[2] - aurocs[3]
ax.annotate(f'Drop: {drop_lstm:+.3f}',
            xy=(0.5, (aurocs[0]+aurocs[1])/2),
            xytext=(0.5, min(aurocs[0],aurocs[1])-0.02),
            ha='center', fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))

plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_race_prediction_auroc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_race_prediction_auroc.png')

---
## 7. Summary

In [ ]:
print('=' * 70)
print('SUMMARY')
print('=' * 70)

print('\n── Experiment 1: Does equalising sample size close the gap? ──')
print(comp_df.to_string(index=False))

print('\n── Experiment 2: Can race be predicted from vital signs? ──')
print(f'  LSTM with mask:          AUROC={auroc_with:.4f}')
print(f'  LSTM without mask:       AUROC={auroc_no:.4f}')
print(f'  Drop from mask removal:  {auroc_with-auroc_no:+.4f}')
print(f'  XGBoost all features:    AUROC={roc_auc_score(y_black_vs_rest, oof_race_xgb_with):.4f}')
print(f'  XGBoost mean only:       AUROC={roc_auc_score(y_black_vs_rest, oof_race_xgb_no):.4f}')

print('\n── Interpretation ──')
if auroc_with > 0.65:
    print('  Race IS predictable from vital signs (AUROC > 0.65).')
    print('  This confirms race is encoded in clinical data.')
    if (auroc_with - auroc_no) > 0.05:
        print('  Large drop when mask removed: monitoring patterns')
        print('  (not physiology) are the primary driver.')
    else:
        print('  Small drop when mask removed: vital sign values themselves')
        print('  also carry racial signal, not just monitoring patterns.')
else:
    print('  Race is not strongly predictable from vital signs alone.')

print(f'\nOutputs saved to: {EXP_DIR}')
for f in sorted(EXP_DIR.glob('*')):
    print(f'  {f.name}')